# PowerNZ En Kaggle

Este notebook es la forma facil de analizar un video sin usar mi PC.

## Lo Que Tengo Que Hacer

1. En Kaggle, abrir este notebook.
2. En `Settings`, activar `Internet`.
3. En `Accelerator`, elegir `GPU T4` o `GPU P100` si aparece.
4. Subir mi video como dataset o adjuntarlo al notebook.
5. Cambiar `EXERCISE` si hace falta.
6. Ejecutar todo.

El video final queda en `/kaggle/working/powernz_analizado.mp4`.

## 1. Configuracion Simple

Cambio solo esto:

- `EXERCISE = "deadlift"` para peso muerto.
- `EXERCISE = "squat"` para sentadilla.
- `EXERCISE = "bench"` para press banca.

Si el plato no calibra bien, ajusto `PLATE_DIAMETER_PX`. Para empezar dejo `120`.

In [ ]:
EXERCISE = "deadlift"  # deadlift, squat o bench
PLATE_DIAMETER_PX = 120
OUTPUT_NAME = "powernz_analizado.mp4"

# Uso esta rama porque aqui esta el descargador de modelos y el notebook.
# Cuando esto se fusione a main, puedo cambiar BRANCH a "main".
REPO_URL = "https://github.com/juanrdzmb/PowerNZ.git"
BRANCH = "codex/model-download-and-launcher"

## 2. Preparar PowerNZ

Esta celda clona el repo, instala dependencias y descarga los modelos desde Hugging Face.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

workdir = Path("/kaggle/working")
repo_dir = workdir / "PowerNZ"

if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)

subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["python", "model_downloader.py"], check=True)

## 3. Encontrar El Video

Busco automaticamente videos dentro de `/kaggle/input`. Si hay varios, uso el primero y muestro la lista.

In [ ]:
from pathlib import Path

video_extensions = ("*.mp4", "*.mov", "*.avi", "*.mkv")
videos = []
for pattern in video_extensions:
    videos.extend(Path("/kaggle/input").rglob(pattern))
videos = sorted(videos)

if not videos:
    raise FileNotFoundError("No encontre videos en /kaggle/input. Sube un video como dataset al notebook.")

for index, path in enumerate(videos):
    print(index, path)

VIDEO_PATH = str(videos[0])
print("\nVoy a analizar:", VIDEO_PATH)

## 4. Analizar

Esta celda ejecuta PowerNZ y guarda el video final.

In [ ]:
from pathlib import Path
import subprocess

output_path = Path("/kaggle/working") / OUTPUT_NAME

command = [
    "python", "main.py",
    "--input", VIDEO_PATH,
    "--output", str(output_path),
    "--exercise", EXERCISE,
    "--pose-backend", "yolo",
    "--plate-diameter-px", str(PLATE_DIAMETER_PX),
    "--output-format", "portrait-720",
    "--no-mobile-conversion",
]

print("Ejecutando:")
print(" ".join(command))
subprocess.run(command, check=True)
print("\nListo:", output_path)

## 5. Descargar Resultado

En Kaggle, miro el panel derecho o la seccion `Output` y descargo:

```text
/kaggle/working/powernz_analizado.mp4
```

Si quiero ver una previsualizacion dentro del notebook, ejecuto la celda de abajo.

In [ ]:
from IPython.display import Video, display

display(Video(str(output_path), embed=False, width=360))